In [8]:
import torch
import torch.nn.functional as F
from torch_geometric.transforms import ToUndirected
from torch_geometric.utils import add_self_loops
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.model_selection import train_test_split

import os
import sys
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))

from utils.graph_utils import load_graph, save_graph
from utils.sample_scoring import process_and_save, do_radical_search, do_biological_logfc, do_std
from data_processing.network_generator import PatientNetworkGenerator, build_knn_graph_with_masks
from hetero_base_models.train_hybridkg import (compute_link_loss, 
                            split_edges,
                            evaluate,
                            evaluate_link,
                            train_one_epoch,
                            train,
                            test,
                            build_x_dict,
                            set_seed
                            )
from hetero_base_models.utilities import (convert_to_hetero_data, 
                                          bridge_names_to_indices,
                                        assign_kg_by_EdgeScore,
                                        get_top_k_assignments,
                                        convert_to_hetero_data, 
                                        bridge_names_to_indices)
from hetero_base_models.base_models import get_model
import argparse

### Configs

In [32]:
config = {
'exp_path': "./data/ADNI/adni_exp_2cls.csv",
'scoring_path': "./data/ADNI/sample_scoring/sample_scoring_ecdf.csv",
'kg_disease_path': '../datasets/base_kgs/adkg_with_isolated_prs.pkl',
'kg_health_path': '../datasets/base_kgs/healthykg_with_isolated_prs.pkl',
'output_dir': "../results/HybridLP/adni/ecdf/",
'model':'gat',
'hidden_channels': 128,
'out_channels': 64,
'num_layers': 3,
'heads': 4,
'dropout': 0.3,
'epochs': 100,
'lr': 1e-3,
'weight_decay': 1e-5,
'lambda_link': 0.7,
'val_ratio': 0.15,
'test_ratio':0.15,
'seed':42,
'device': 'cuda',
'assign_method':'cls',
'confidence': 0.6
}
# convert config to Namespace
args = argparse.Namespace(**config)

### main() script

In [4]:
# main() script

print("\nStart Loading Required Data...\n")
# Load expression df, smaple scoring df, KG
exp_df = pd.read_csv(args.exp_path, index_col=0)
if exp_df.shape[0] > exp_df.shape[1]:
    exp_df = exp_df.transpose()
scoring_df = pd.read_csv(args.scoring_path, index_col=0)
# clean exp_df before K-NN
# drop genes with no variation
exp_df = exp_df.loc[:, exp_df.std() > 0]
# Using median is usually safer for gene expression
exp_df = exp_df.fillna(exp_df.median())
# normalize safely
min_val = exp_df.min()
max_val = exp_df.max()
exp_norm = (exp_df - min_val) / (max_val - min_val + 1e-9)
# final fill-na
exp_norm = exp_norm.fillna(0)
# load kgs
kg_disease = load_graph(args.kg_disease_path)
kg_control = load_graph(args.kg_health_path)

pattern_disease =  r'^p\(HGNC:"([^"]+)"\)$'
pattern_control =  r'^p\(UniProtKB:"([^"_%]+)_[A-Z]+"\)$'

# 1. Generate NetworkX Graph
print("\nStart Generating Disease-Sample-Healthy Network...\n")
png = PatientNetworkGenerator(kg_disease, kg_control)
full_graph, summary_df, radicals, nhs_scores = png.generate_hybrid_network(
                                data=scoring_df,
                                exp_df=exp_norm,
                                pattern_disease=pattern_disease,
                                pattern_control=pattern_control,
                                disease_label=1,
                                control_label=0
                                )

# 2. Get biological mappings (Names)
masked_names, d_up_names, d_down_names, c_up_names, c_down_names = png.get_candidate_node_names(
                                                            summary_df, 
                                                            radicals, 
                                                            pattern_disease, 
                                                            pattern_control)

# 3. Convert to PyG (This creates the node_mappings)
print("\nStart Converting to HeteroData...\n")
data, node_mappings = convert_to_hetero_data(full_graph)

# 4. Bridge to Indices
val_indices, d_up_ids, d_down_ids, c_up_ids, c_down_ids = bridge_names_to_indices(
    masked_names, d_up_names, d_down_names, c_up_names, c_down_names, node_mappings
)
# 5. Training Pipline
print("\nPrepare for training...\n")
# Setup
set_seed(args.seed)
device = torch.device(args.device if torch.cuda.is_available() else "cpu")

# Build features
data.x_dict = build_x_dict(data)

# (2) Edge split
edge_index_dict = {
    etype: data[etype].edge_index
    for etype in data.edge_types
}
train_edges, val_edges, test_edges = split_edges(
    edge_index_dict,
    val_ratio=args.val_ratio,
    test_ratio=args.test_ratio,
    seed=args.seed
)
# Labels
y = data["Patient"].y
num_classes = int(y.max().item() + 1) if y.dim() == 1 else y.size(-1)
print(f"Number of classes: {num_classes}")

# (3) Build model
print("\nStart building model...\n")
model = get_model(
    data=data,
    model_type=args.model,
    hidden_channels=args.hidden_channels,
    out_channels=args.out_channels,
    num_layers=args.num_layers,
    heads=args.heads,
    dropout=args.dropout,
    num_classes=num_classes,
    device=device
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=args.lr,
    weight_decay=args.weight_decay
)


Start Loading Required Data...

Loaded graph from ../datasets/base_kgs/adkg_with_isolated_prs.pkl: 3988 nodes, 10554 edges
Loaded graph from ../datasets/base_kgs/healthykg_with_isolated_prs.pkl: 4787 nodes, 13775 edges

Start Generating Disease-Sample-Healthy Network...

Contructing Patient-Patient Netwrok


Rewiring: 100%|██████████| 455/455 [00:00<00:00, 622.97it/s]


Rewired 0 edges.
Calculating Neighborhood Affinity Scores (NAS)...


Linking Samples to KGs: 100%|██████████| 37310/37310 [00:04<00:00, 7546.78it/s]



Start Converting to HeteroData...

Starting conversion from NetworkX to HeteroData...


/Users/yuxiaoxuan/master_thesis/HybridKG/hetero_base_models/utilities.py:40: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1719361045918/work/torch/csrc/utils/tensor_new.cpp:277.)
  patient_x = torch.tensor([G.nodes[pid]['x'] for pid in p_ids], dtype=torch.float)


HeteroData created: 25 node types, 816 edge types.

Prepare for training...

Number of classes: 2

Start building model...



In [33]:
# Initial training
print("\nStart training...\n")
print("--- Stage 1: Initial Training ---")
model, history = train(model, data, train_edges, None, optimizer, device, epochs=args.epochs)


# --- STAGE 2: Inference & Assignment ---
print("--- Stage 2: Augmenting Graph with Validation Nodes ---")
model.eval()
with torch.no_grad():
    x_dict = {k: v.to(device) for k, v in data.x_dict.items()}
    z_dict = model.encode(x_dict, train_edges)
    
    d_scores_dict, c_scores_dict = assign_kg_by_EdgeScore(
        model, z_dict, val_indices, 
        d_up_ids, d_down_ids, c_up_ids, c_down_ids, 
        device
    )


Start training...

--- Stage 1: Initial Training ---


Training HeteoGNN:   1%|          | 1/100 [00:20<33:36, 20.36s/it]

Epoch 0 | {'loss': 21.267154693603516, 'cls_loss': 0.8099602460861206, 'link_loss': 41.72434997558594} | Val {'acc': 0.47058823529411764, 'f1_macro': 0.32, 'auroc': 0.3819444444444445, 'auprc': 0.49309513603098193}


Training HeteoGNN:   2%|▏         | 2/100 [00:35<28:11, 17.26s/it]

Epoch 1 | {'loss': 12.505219459533691, 'cls_loss': 0.7875073552131653, 'link_loss': 24.222930908203125} | Val {'acc': 0.47058823529411764, 'f1_macro': 0.32, 'auroc': 0.5, 'auprc': 0.54023671465211}


...

Training HeteoGNN:  99%|█████████▉| 99/100 [23:54<00:14, 14.40s/it]

Epoch 98 | {'loss': 0.3260718286037445, 'cls_loss': 0.0011635718401521444, 'link_loss': 0.6509801149368286} | Val {'acc': 0.6176470588235294, 'f1_macro': 0.6146469049694856, 'auroc': 0.6336805555555555, 'auprc': 0.6530427058503543}


Training HeteoGNN: 100%|██████████| 100/100 [24:09<00:00, 14.49s/it]

Epoch 99 | {'loss': 0.3187837302684784, 'cls_loss': 0.0036871812772005796, 'link_loss': 0.6338802576065063} | Val {'acc': 0.6323529411764706, 'f1_macro': 0.6257979308826767, 'auroc': 0.6440972222222221, 'auprc': 0.6526266912256204}


--- Stage 2: Augmenting Graph with Validation Nodes ---


In [ ]:
def get_top_k_assignments(d_scores_dict, c_scores_dict, d_ids, c_ids, node_mappings):
    """
    Ranks proteins based on combined scores from two dictionaries and returns assignments.
    """
    assignments = {}
        
    # Iterate through each sample_node_id
    for sample_id in d_scores_dict.keys():
        assignments[sample_id] = {}
        
        # Process 'up' and 'down' relations
        for relation in ['up', 'down']:
            # 1. Map scores to their protein IDs
            # a dictionary of protein_id -> d_score/c_score
            protein_map = {}
            
            # Extract d_scores
            d_scores = d_scores_dict[sample_id][relation]
            # normalize d_scores
            d_scores = (d_scores - d_scores.mean())/(d_scores.std()+1e-6)

            d_indices = d_ids[sample_id][relation]
            for idx, prot_id in enumerate(d_indices):
                protein_map[prot_id] = [d_scores[idx].item(),'d']
                
            # Extract c_scores and update map
            c_scores = c_scores_dict[sample_id][relation]
            # normalize c_scores
            c_scores = (c_scores - c_scores.mean())/(c_scores.std()+1e-6)

            c_indices = c_ids[sample_id][relation]
            for idx, prot_id in enumerate(c_indices):
                if prot_id in protein_map:
                    print(f'protein node id {prot_id} exists in both kgs candidate protein list.')
                else:
                    protein_map[prot_id] = [c_scores[idx].item(),'c']
            # print(d_indices)
            # print(c_indices)
            # print(protein_map)
            # 2. Ranking:by triple score
            sorted_proteins = sorted(
                protein_map.items(), 
                key=lambda item: (item[1][0]), 
                reverse=True
            )
            #print('sorted proteins:\n',sorted_proteins)
            # 3. Slice the Top K
            k = min(len(d_indices), len(c_indices))
            top_k = sorted_proteins[:k]
            #print('topk (sample-relation-protein) scores\n', top_k)
        #break
            # Prepare output tensors
            top_ids = torch.tensor([item[0] for item in top_k])
            top_scores = [item[1] for item in top_k] # [score, d/c] pairs
            
            assignments[sample_id][relation] = {
                'protein_ids': top_ids,
                'scores': top_scores
            }
    return assignments

In [35]:
d_ids = {}
c_ids = {}
for k in d_up_ids.keys():
    d_ids[k] = {}
    d_ids[k]['up'] = d_up_ids[k]
    d_ids[k]['down'] = d_down_ids[k]
for k in c_up_ids.keys():
    c_ids[k] = {}
    c_ids[k]['up'] = c_up_ids[k]
    c_ids[k]['down'] = c_down_ids[k]

assignments = get_top_k_assignments(d_scores_dict,
                                    c_scores_dict,
                                    d_ids,
                                    c_ids,
                                    )

In [36]:
assignments[0]
assignment_logs = []
for sample_id, relations in assignments.items():
    for relation_type, content in relations.items():
        assigned_ids = content['protein_ids'] 
        raw_scores = content['scores']    # List of [score, 'd'/'c']

        for i, prot_id in enumerate(assigned_ids):
            p_id = prot_id.item()
    
            # save edge_log infos
            assignment_logs.append({
                'sample_id': int(sample_id),
                'protein_id': int(p_id),
                'relation': relation_type,
                'score': raw_scores[i][0],
                'source_kg': raw_scores[i][1],
                'label': data['Patient'].y[sample_id].item()
            })

df_logs = pd.DataFrame(assignment_logs)

In [37]:
def calculate_source_ratio(df):
    results = []
    
    # Grouping by sample_id
    grouped = df.groupby('sample_id')
    
    for sample_id, group in grouped:
        num_c = (group['source_kg'] == 'c').sum()
        num_d = (group['source_kg'] == 'd').sum()
        
        
        # Calculate ratio (c / d)
        ratio = num_d / num_c if num_c > 0 else float('inf')
        
        results.append({
            'sample_id': sample_id,
            'c_count': num_c,
            'd_count': num_d,
            'd_c_ratio': ratio,
            # 'nhs_score':group['nhs_score'].mean(),
            'label': group['label'].iloc[0] # To verify alignment with original label
        })
    
    return pd.DataFrame(results)

In [38]:
df_logs_analysis = calculate_source_ratio(df_logs)
df_logs_analysis['d_c_label'] = df_logs_analysis['d_c_ratio'].apply(lambda x: 1 if x > 1 else 0)
df_logs_analysis

,sample_id,c_count,d_count,d_c_ratio,label,d_c_label
0,0,52,20,0.384615,0,0
1,2,30,25,0.833333,1,0
2,5,179,99,0.553073,0,0
3,17,36,18,0.500000,1,0
4,22,49,49,1.000000,0,0
...,...,...,...,...,...,...
132,442,48,28,0.583333,0,0
133,447,61,29,0.475410,1,0
134,449,20,21,1.050000,1,1
135,451,53,48,0.905660,1,0


In [39]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

def assignment_prediction_metrics(df, true_label:str, predicted_label:str, score_label:str):
    df[predicted_label] = df[score_label].apply(lambda x: 1 if x > 1 else 0)
    # 1. Define your ground truth and predictions
    y_true = df[true_label]
    y_pred = df[predicted_label]  # Binary labels (0 or 1)
    y_score = df[score_label] # Continuous scores (0 to 1)

    # 2. Calculate Metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)

    metrics = {
        "Accuracy": accuracy,
        'F1_score':f1,
        'AUROC':auroc,
        'AUPRC':auprc
    }
    return metrics

In [40]:
assignment_prediction_metrics(df=df_logs_analysis,
                            true_label='label', 
                            predicted_label='d_c_label', 
                            score_label='d_c_ratio')

{'Accuracy': 0.4233576642335766,
 'F1_score': 0.2882882882882883,
 'AUROC': 0.4872645547945206,
 'AUPRC': 0.4971106687275819}